<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_11_Hybrid_Neyman_Construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 11 — A simulator-calibrated hybrid Neyman construction

Exercise 5 ended with pseudo-experiments from the learned hNDE model and the sampling distribution of the profile-likelihood-ratio statistic. We now amortize that construction over the signal strength $\mu$.

The notebook builds three increasingly physical descriptions of the conditional test-statistic distribution:

1. draw $500{,}000$ hNDE pseudo-experiments with $\mu\sim\pi(\mu)=\mathrm{Uniform}(0,3)$ and calculate $t_\mu$;
2. train a conditional rational-quadratic-spline flow $q_\phi(t\mid\mu)$;
3. correct it with a matched conditional density ratio $r_1(t,\mu)$, giving the hNDE-level hybrid distribution;
4. draw $100{,}000$ pseudo-experiments from an independent simulator template and learn a second, importance-weighted correction $r_2(t,\mu)$;
5. integrate the one-dimensional hybrid densities explicitly, compare their conditional quantiles, and extract $c_{0.95}(\mu)$;
6. generate another $100{,}000$ untouched simulator pseudo-experiments and train an LF2I-style BCE auditor for the conditional coverage.

The expensive Exercise 5 event networks are frozen and loaded from their checkpoints. All toy ensembles are generated in resumable shards, so a Colab interruption does not discard completed fits.


In [ ]:
## ==========================================================================
# Google Colab setup — run me first. Safe to re-run; a no-op off Colab.
# ==========================================================================
import os, sys

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
N_BKG, N_SIG = 100_000_000, 20_000_000
USE_DRIVE = True
REMAKE_EVENTS = False

import subprocess
from pathlib import Path

DEPENDENCIES = [
    "pytorch-lightning",
    "onnx",
    "onnxruntime",
    "onnxscript",
    "iminuit",
    "mplhep",
    "nflows",
    "pyarrow",
]


def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab")
    else:
        ROOT = Path("/content")
    ROOT.mkdir(parents=True, exist_ok=True)

    REPO_DIR = ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    WORK_DIR = REPO_DIR / "workshops" / "ml4hep_tifr"

    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)

    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )
    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        import_path = str(import_dir.resolve())
        if import_path not in sys.path:
            sys.path.insert(0, import_path)
    run(sys.executable, "-m", "pip", "install", "-q", *DEPENDENCIES)

    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)
    if REMAKE_EVENTS or not Path("dataframes/signal.parquet").exists():
        run(
            sys.executable,
            TUTORIAL_DIR / "generate_distributions.py",
            "--n_bkg", N_BKG,
            "--n_sig", N_SIG,
        )

print("Working dir:", os.getcwd())


## Statistical construction

For a pseudo-dataset $\mathcal D$ and a tested value $\mu$, Exercise 5 uses

$$
t_\mu(\mathcal D)=-2\log\frac{L(\mu;\mathcal D)}{L(\widehat\mu;\mathcal D)}\geq0.
$$

Let $p_{\rm H}(t\mid\mu)$ denote toys generated by the frozen hNDE model and $p_{\rm sim}(t\mid\mu)$ toys generated from the independent simulator template. A conditional flow first learns $q_\phi(t\mid\mu)$. In the transformed coordinate $y=\log(t+\epsilon)$, its tiny unphysical tail below $y_{\min}=\log\epsilon$ is removed by conditional rejection sampling. We write $\widetilde q_\phi=q_\phi\,\mathbb I[y\geq y_{\min}]/C_\phi(\mu)$ for that physical reference. A balanced matched-pair classifier then estimates

$$
r_1(t,\mu)=\frac{p_{\rm H}(t\mid\mu)}{\widetilde q_\phi(t\mid\mu)},
\qquad
p_1(t\mid\mu)=\frac{\widetilde q_\phi(t\mid\mu)r_1(t,\mu)}{\widetilde Z_1(\mu)}.
$$

The simulator correction compares positive samples from $p_{\rm sim}$ with physical-reference samples weighted by $r_1/\widetilde Z_1$. Its weighted BCE is

$$
\mathcal L_2=
-\mathbb E_{\pi(\mu)p_{\rm sim}}\log D_2
-\mathbb E_{\pi(\mu)\widetilde q_\phi}
\left[\frac{r_1}{\widetilde Z_1(\mu)}\log(1-D_2)\right],
$$

so the optimal odds are $r_2=p_{\rm sim}/p_1$. The final conditional model is therefore

$$
p_2(t\mid\mu)=
\frac{\widetilde q_\phi(t\mid\mu)r_1(t,\mu)r_2(t,\mu)}{\widetilde Z_2(\mu)}.
$$

Because large $t_\mu$ rejects the tested value, the 95% acceptance cutoff is the **upper** conditional quantile $c_{0.95}(\mu)=F^{-1}(0.95\mid\mu)$. This reverses the lower-quantile sign convention used for the log-likelihood-ratio statistic in the [LF2I paper](https://arxiv.org/abs/2107.03920).


In [ ]:
import gc
import os
from pathlib import Path

# JAX fits the toy batches before PyTorch trains the large spline. Avoid
# reserving the whole Colab GPU so both frameworks can share it.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
from scipy.interpolate import PchipInterpolator
from scipy.special import logsumexp
from scipy.stats import chi2

import torch

from nsbi_common_utils.training.utils import load_trained_model
from utils import FEATURES, predict_with_model
from utils_hnpe import (
    ratio_classifier_ensemble_logit,
    train_ratio_classifier,
    train_spline_flow,
)
from utils_neyman import (
    asimov_test_statistic,
    binned_coverage,
    build_compressed_q_model,
    conditional_density_grid,
    conditional_quantiles,
    coverage_auditor_probability,
    importance_effective_sample_size,
    run_cached_toy_ensemble,
    sample_truncated_spline_flow,
    simulator_templates_from_exercise5,
    train_coverage_auditor,
    wilson_interval,
)
from utils_nf import (
    accumulate_preselection_histogram,
    checkpoint_path,
    choose_preselection_ratio_cut,
    flow_sample_x,
    load_flow,
)
from utils_plotting import export_standalone_figure_script

FEATURES = list(FEATURES)
SEED = 11082026
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Configuration and frozen Exercise 5 inputs

`FAST_MODE=True` is a structural test. The default is the requested construction: 500,000 hNDE toys, 100,000 simulator-calibration toys, and a separate 100,000-toy audit. The flow/ratio checkpoints and toy shards persist in Drive.

The statistic flow uses the Exercise 5 rational-quadratic-spline settings: ten spline transforms, 16 bins, tail bound 5, four residual blocks, and width 1024. For a scalar target, `utils_hnpe` uses the mathematically appropriate one-dimensional autoregressive spline; a coupling layer would otherwise have no second coordinate to condition on.


In [ ]:
BASE_PATH = Path("dataframes")
PRESEL_MODEL_DIR = Path("models_PRESEL")
REFERENCE_FLOW_MODEL_DIR = Path("models_flows_hybrid_reference_spline16_tail5")
RATIO_MODEL_DIR = {
    "signal": Path("models_Hybrid_SigvsRef_5M_ensemble4"),
    "background": Path("models_Hybrid_BkgvsRef_5M_ensemble4"),
}
HYBRID_DENSITY_DIR = Path("saved_densities_hybrid")

FAST_MODE = False
LOAD_IF_AVAILABLE = True
RUN_TAG = "fast" if FAST_MODE else "full_v1"
MODEL_DIR = Path("models_exercise11_hybrid_neyman_v1") / RUN_TAG
CACHE_DIR = Path("saved_exercise11_hybrid_neyman_v1") / RUN_TAG
PLOT_DIR = Path("plots_exercise11_hybrid_neyman_v1") / RUN_TAG
FIGURE_SCRIPT_DIR = Path("exercise11_figures_scripts")
RATIO1_MODEL_DIR = MODEL_DIR / "hnde_residual_ensemble4"
RATIO2_MODEL_DIR = MODEL_DIR / "simulator_residual_ensemble4"
for directory in [
    MODEL_DIR, CACHE_DIR, PLOT_DIR, FIGURE_SCRIPT_DIR,
    RATIO1_MODEL_DIR, RATIO2_MODEL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

SAMPLE_PATHS = {
    "signal": BASE_PATH / "signal.parquet",
    "background": BASE_PATH / "background.parquet",
}
SPLIT_SEED = 0
PRESEL_TRAIN_FRACTION = 0.5
FLOW_TRAIN_FRACTION = 0.92
STREAM_BATCH_SIZE = 100_000
PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO = 250.0
PRESEL_CUT_HISTOGRAM_BINS = 4_000
PRESEL_LOG_RATIO_RANGE = (-20.0, 20.0)
REFERENCE_FLOW_TYPE = "quadratic_spline"
REFERENCE_SAMPLING_BATCH_SIZE = 65_536
RATIO_ENSEMBLE_SIZE = 4
RATIO_EVALUATION_BATCH_SIZE = 100_000
RATIO_FLOOR = 1.0e-12

MU_RANGE = (0.0, 3.0)
TOY_Q_BINS = 512
TOY_BATCH_SIZE = 2_000
TOY_NEWTON_STEPS = 16
TOY_MU_MAX = 12.0
TOY_FIT_FINGERPRINT = "jax_newton16_bisection64_mumax12_v1"
T_OFFSET = 1.0e-6
LOG_RATIO_CLIP = 15.0
QUANTILE_LEVELS = np.asarray([0.50, 0.68, 0.90, 0.95, 0.99])
ANCHOR_MUS = np.asarray([0.0, 3.0])

if FAST_MODE:
    N_REFERENCE_EVENTS = 250_000
    N_HNDE_TOYS = 50_000
    N_FLOW_TOYS = 40_000
    N_RATIO1_TOYS = 10_000
    N_SIMULATOR_CALIBRATION_TOYS = 20_000
    N_SIMULATOR_AUDIT_TOYS = 20_000
    N_ANCHOR_TOYS = 5_000
    N_AUDIT_ANCHOR_TOYS = 5_000
    FLOW_EPOCHS = 8
    RATIO_EPOCHS = 12
    QUADRATURE_MU_POINTS = 101
    QUADRATURE_Y_POINTS = 1_024
else:
    N_REFERENCE_EVENTS = 5_000_000
    N_HNDE_TOYS = 500_000
    N_FLOW_TOYS = 400_000
    N_RATIO1_TOYS = 100_000
    N_SIMULATOR_CALIBRATION_TOYS = 100_000
    N_SIMULATOR_AUDIT_TOYS = 100_000
    N_ANCHOR_TOYS = 25_000
    N_AUDIT_ANCHOR_TOYS = 25_000
    FLOW_EPOCHS = 70
    RATIO_EPOCHS = 50
    QUADRATURE_MU_POINTS = 301
    QUADRATURE_Y_POINTS = 4_096

if N_FLOW_TOYS + N_RATIO1_TOYS != N_HNDE_TOYS:
    raise ValueError("The disjoint hNDE flow/ratio splits must exhaust the toys.")

STATISTIC_FLOW_MODEL_CONFIG = {
    "n_coupling_layers": 10,
    "hidden_features": 1024,
    "hidden_layers": 4,
    "spline_num_bins": 16,
    "spline_tail_bound": 5.0,
    "dropout_probability": 0.0,
}
STATISTIC_FLOW_TRAINING_CONFIG = {
    "batch_size": 2048,
    "n_epochs": FLOW_EPOCHS,
    "learning_rate": 1.0e-4,
    "lr_scheduler_factor": 0.2,
    "lr_scheduler_patience": 2,
    "min_learning_rate": 1.0e-7,
    "weight_decay": 0.0,
    "validation_fraction": 0.20,
    "patience": 5,
    "gradient_clip": 5.0,
}
CORRECTION_MODEL_CONFIG = {
    "hidden_features": 1024,
    "hidden_layers": 4,
    "dropout_probability": 0.0,
}
CORRECTION_TRAINING_CONFIG = {
    "batch_size": 2048,
    "n_epochs": RATIO_EPOCHS,
    "learning_rate": 1.0e-3,
    "lr_scheduler": "step",
    "lr_scheduler_factor": 0.01,
    "lr_scheduler_patience": 10,
    "validation_fraction": 0.20,
    "patience": 20,
    "gradient_clip": 5.0,
}
AUDITOR_MODEL_CONFIG = {"hidden_features": 64, "hidden_layers": 3}
AUDITOR_TRAINING_CONFIG = {
    "batch_size": 2048,
    "n_epochs": 200 if not FAST_MODE else 40,
    "learning_rate": 1.0e-3,
    "weight_decay": 1.0e-4,
    "lr_scheduler_factor": 0.3,
    "lr_scheduler_patience": 5,
    "min_learning_rate": 1.0e-6,
    "validation_fraction": 0.20,
    "patience": 20,
    "gradient_clip": 5.0,
}


def export_exercise11_figure(fig, script_name):
    fig.savefig(PLOT_DIR / f"{script_name}.png", dpi=160)
    return export_standalone_figure_script(
        fig, script_name=script_name, output_dir=FIGURE_SCRIPT_DIR
    )


required_paths = [
    PRESEL_MODEL_DIR / "model0.onnx",
    PRESEL_MODEL_DIR / "model_scaler0.bin",
    checkpoint_path("reference", REFERENCE_FLOW_MODEL_DIR, REFERENCE_FLOW_TYPE),
    HYBRID_DENSITY_DIR / "weights_asimov.npy",
    HYBRID_DENSITY_DIR / "ratio_signal_asimov.npy",
    HYBRID_DENSITY_DIR / "ratio_background_asimov.npy",
]
for sample_name, model_dir in RATIO_MODEL_DIR.items():
    for member in range(RATIO_ENSEMBLE_SIZE):
        required_paths.extend([
            model_dir / f"model{member}.onnx",
            model_dir / f"model_scaler{member}.bin",
        ])
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Exercise 11 loads the frozen Exercise 5 model and held-out arrays. "
        "Run Exercise 5 through 'Reconstruct and validate the hybrid densities' "
        "first. Missing:\n" + "\n".join(f"  - {path}" for path in missing_paths)
    )

print(f"Run tag: {RUN_TAG}")
print(f"hNDE toys: {N_HNDE_TOYS:,}")
print(f"simulator calibration/audit toys: "
      f"{N_SIMULATOR_CALIBRATION_TOYS:,}/{N_SIMULATOR_AUDIT_TOYS:,}")
print(f"Standalone figure scripts: {FIGURE_SCRIPT_DIR}/")


## Load the frozen hNDE event model

The PRESEL classifier, post-selection yields, reference flow, and the two four-member density-ratio ensembles are the same objects used by Exercise 5. No event-level model is retrained here.


In [ ]:
def as_inference_session(model_candidate):
    if isinstance(model_candidate, ort.InferenceSession):
        return model_candidate
    available = ort.get_available_providers()
    providers = [
        provider
        for provider in ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if provider in available
    ] or available
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    options.inter_op_num_threads = 1
    return ort.InferenceSession(
        model_candidate.SerializeToString(),
        sess_options=options,
        providers=providers,
    )


PRESEL_scaler, PRESEL_model_proto = load_trained_model(
    PRESEL_MODEL_DIR / "model0.onnx",
    PRESEL_MODEL_DIR / "model_scaler0.bin",
)
PRESEL_model = as_inference_session(PRESEL_model_proto)
del PRESEL_model_proto


def evaluate_PRESEL_ratio(feature_dataframe):
    ratio = predict_with_model(
        feature_dataframe.astype("float32", copy=False),
        scaler=PRESEL_scaler,
        model=PRESEL_model,
    )
    return np.asarray(ratio, dtype=np.float64).reshape(-1)


PRESEL_STATE_CANDIDATES = [
    CACHE_DIR / "exercise5_preselection_state.npz",
    Path("saved_asimov_nis_influence_v2/exercise5_preselection_state.npz"),
    Path("saved_exercise7_misspecification/exercise5_preselection_state.npz"),
]
existing_state = next(
    (path for path in PRESEL_STATE_CANDIDATES if path.exists()), None
)
if existing_state is not None:
    state = np.load(existing_state)
    PRESEL_RATIO_CUT = float(state["ratio_cut"])
    LAM_SIG = float(state["lambda_signal"])
    LAM_BKG = float(state["lambda_background"])
    print(f"Loaded PRESEL state from {existing_state}")
else:
    edges = np.linspace(
        PRESEL_LOG_RATIO_RANGE[0], PRESEL_LOG_RATIO_RANGE[1],
        PRESEL_CUT_HISTOGRAM_BINS + 1,
    )
    histograms, statistics = {}, {}
    for sample_name in ["signal", "background"]:
        histograms[sample_name], statistics[sample_name] = (
            accumulate_preselection_histogram(
                SAMPLE_PATHS[sample_name],
                features=FEATURES,
                ratio_predictor=evaluate_PRESEL_ratio,
                log_ratio_edges=edges,
                batch_size=STREAM_BATCH_SIZE,
                presel_fraction=PRESEL_TRAIN_FRACTION,
                flow_train_fraction=FLOW_TRAIN_FRACTION,
                split_seed=SPLIT_SEED,
            )
        )
    PRESEL_RATIO_CUT, diagnostics = choose_preselection_ratio_cut(
        histograms["signal"], histograms["background"], edges,
        signal_inclusive_yield=statistics["signal"]["inclusive_weight"],
        background_inclusive_yield=statistics["background"]["inclusive_weight"],
        signal_partition_weight=statistics["signal"]["partition_weight"],
        background_partition_weight=statistics["background"]["partition_weight"],
        target_background_to_signal=PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO,
    )
    LAM_SIG = diagnostics["histogram_signal_yield"]
    LAM_BKG = diagnostics["histogram_background_yield"]
    np.savez(
        PRESEL_STATE_CANDIDATES[0],
        ratio_cut=PRESEL_RATIO_CUT,
        lambda_signal=LAM_SIG,
        lambda_background=LAM_BKG,
    )

reference_flow = load_flow(
    "reference",
    model_dir=REFERENCE_FLOW_MODEL_DIR,
    flow_type=REFERENCE_FLOW_TYPE,
    device=device,
    expected_features=FEATURES,
)
ratio_models = {}
for sample_name, model_dir in RATIO_MODEL_DIR.items():
    ratio_models[sample_name] = []
    for member in range(RATIO_ENSEMBLE_SIZE):
        scaler, model_proto = load_trained_model(
            model_dir / f"model{member}.onnx",
            model_dir / f"model_scaler{member}.bin",
        )
        ratio_models[sample_name].append(
            {"scaler": scaler, "model": as_inference_session(model_proto)}
        )

print(f"PRESEL ratio cut: {PRESEL_RATIO_CUT:.6g}")
print(f"Post-selection yields: signal={LAM_SIG:.6g}, background={LAM_BKG:.6g}")


## Reconstruct and cache the compressed Exercise 5 likelihood

For every event, the fitted likelihood depends only on

$$
q(x)=\frac{\lambda_S r_S(x)}{\lambda_B r_B(x)}.
$$

We draw the same five-million-event reference sample as Exercise 5, normalize both process ratios on it, and compress $\log q$ into 512 bins. The compressed signal/background probabilities generate hNDE toys; their ratio defines the **frozen likelihood** used to fit both hNDE and simulator toys.


In [ ]:
def evaluate_ratio(sample_name, values, batch_size=RATIO_EVALUATION_BATCH_SIZE):
    values = np.asarray(values, dtype=np.float32)
    chunks = []
    for start in range(0, len(values), int(batch_size)):
        batch = pd.DataFrame(
            values[start : start + int(batch_size)], columns=FEATURES
        )
        member_predictions = []
        for pack in ratio_models[sample_name]:
            prediction = predict_with_model(
                batch, scaler=pack["scaler"], model=pack["model"]
            )
            member_predictions.append(
                np.asarray(prediction, dtype=np.float64).reshape(-1)
            )
        chunks.append(np.mean(np.stack(member_predictions, axis=0), axis=0))
    ratio = np.concatenate(chunks) if chunks else np.empty(0)
    if not np.isfinite(ratio).all():
        raise FloatingPointError(f"Non-finite {sample_name} ratio.")
    return np.maximum(ratio, RATIO_FLOOR)


def sample_preselected_flow(flow_pack, n_events, batch_size=65_536):
    accepted_chunks = []
    n_kept = 0
    n_generated = 0
    n_passed = 0
    while n_kept < int(n_events):
        needed = int(n_events) - n_kept
        current_batch = max(int(batch_size), min(4 * int(batch_size), 2 * needed))
        generated = flow_sample_x(flow_pack, current_batch, batch_size=batch_size)
        generated_df = pd.DataFrame(generated, columns=FEATURES)
        passes = evaluate_PRESEL_ratio(generated_df) >= PRESEL_RATIO_CUT
        n_generated += len(generated)
        n_passed += int(passes.sum())
        if np.any(passes):
            accepted_chunks.append(generated[passes])
            n_kept += int(passes.sum())
    accepted = np.concatenate(accepted_chunks, axis=0)[: int(n_events)]
    return accepted.astype(np.float32, copy=False), n_passed / n_generated


COMPRESSED_MODEL_PATH = CACHE_DIR / "compressed_exercise5_model.npz"
compression_cache_metadata = {
    "cache_version": 1,
    "n_reference_events": N_REFERENCE_EVENTS,
    "toy_q_bins": TOY_Q_BINS,
    "lambda_signal": LAM_SIG,
    "lambda_background": LAM_BKG,
    "presel_ratio_cut": PRESEL_RATIO_CUT,
}
if COMPRESSED_MODEL_PATH.exists():
    saved = np.load(COMPRESSED_MODEL_PATH)
    missing_metadata = set(compression_cache_metadata) - set(saved.files)
    mismatched_metadata = [
        name for name, expected in compression_cache_metadata.items()
        if name in saved.files
        and not np.isclose(float(saved[name]), float(expected), rtol=1e-12)
    ]
    if missing_metadata or mismatched_metadata:
        raise RuntimeError(
            "The compressed-model cache has stale provenance. Bump "
            "RUN_TAG (recommended) or remove only that versioned cache. "
            f"Missing={sorted(missing_metadata)}, "
            f"mismatched={mismatched_metadata}."
        )
    COMPRESSED_Q = saved["q"]
    HNDE_SIGNAL_PROBABILITY = saved["signal_probability"]
    HNDE_BACKGROUND_PROBABILITY = saved["background_probability"]
    LOG_Q_EDGES = saved["log_q_edges"]
    RATIO_NORMALIZATION = {
        "signal": float(saved["signal_normalization"]),
        "background": float(saved["background_normalization"]),
    }
    compression_truth = saved["validation_truth"]
    compression_test = saved["validation_test"]
    compression_unbinned = saved["validation_unbinned"]
    compression_binned = saved["validation_binned"]
    saved.close()
    print(f"Loaded compressed hNDE model from {COMPRESSED_MODEL_PATH}")
else:
    torch.manual_seed(SEED + 100)
    reference_values, reference_acceptance = sample_preselected_flow(
        reference_flow, N_REFERENCE_EVENTS, REFERENCE_SAMPLING_BATCH_SIZE
    )
    raw_signal = evaluate_ratio("signal", reference_values)
    raw_background = evaluate_ratio("background", reference_values)
    RATIO_NORMALIZATION = {
        "signal": float(raw_signal.mean()),
        "background": float(raw_background.mean()),
    }
    ratio_signal = raw_signal / RATIO_NORMALIZATION["signal"]
    ratio_background = raw_background / RATIO_NORMALIZATION["background"]
    weight_signal = ratio_signal / N_REFERENCE_EVENTS
    weight_background = ratio_background / N_REFERENCE_EVENTS
    event_q = (
        LAM_SIG / LAM_BKG * ratio_signal / ratio_background
    )
    compressed = build_compressed_q_model(
        event_q,
        weight_signal,
        weight_background,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        n_bins=TOY_Q_BINS,
    )
    COMPRESSED_Q = compressed["q"]
    HNDE_SIGNAL_PROBABILITY = compressed["signal_probability"]
    HNDE_BACKGROUND_PROBABILITY = compressed["background_probability"]
    LOG_Q_EDGES = compressed["log_q_edges"]

    compression_rows = []
    for truth_mu in [0.0, 0.25, 1.0, 2.0, 3.0]:
        unbinned_expected = (
            truth_mu * LAM_SIG * weight_signal
            + LAM_BKG * weight_background
        )
        binned_expected = (
            truth_mu * LAM_SIG * HNDE_SIGNAL_PROBABILITY
            + LAM_BKG * HNDE_BACKGROUND_PROBABILITY
        )
        for test_mu in [0.0, 0.5, 1.0, 2.0, 3.0]:
            if test_mu == truth_mu:
                continue
            compression_rows.append((
                truth_mu,
                test_mu,
                asimov_test_statistic(
                    test_mu, truth_mu, event_q, unbinned_expected,
                    lam_signal=LAM_SIG,
                ),
                asimov_test_statistic(
                    test_mu, truth_mu, COMPRESSED_Q, binned_expected,
                    lam_signal=LAM_SIG,
                ),
            ))
    compression_rows = np.asarray(compression_rows, dtype=np.float64)
    compression_truth = compression_rows[:, 0]
    compression_test = compression_rows[:, 1]
    compression_unbinned = compression_rows[:, 2]
    compression_binned = compression_rows[:, 3]
    np.savez_compressed(
        COMPRESSED_MODEL_PATH,
        q=COMPRESSED_Q,
        signal_probability=HNDE_SIGNAL_PROBABILITY,
        background_probability=HNDE_BACKGROUND_PROBABILITY,
        log_q_edges=LOG_Q_EDGES,
        signal_normalization=RATIO_NORMALIZATION["signal"],
        background_normalization=RATIO_NORMALIZATION["background"],
        validation_truth=compression_truth,
        validation_test=compression_test,
        validation_unbinned=compression_unbinned,
        validation_binned=compression_binned,
        **compression_cache_metadata,
    )
    print(f"Saved compressed hNDE model to {COMPRESSED_MODEL_PATH}")
    print(f"Reference PRESEL acceptance: {reference_acceptance:.3%}")
    del reference_values, raw_signal, raw_background
    del ratio_signal, ratio_background, weight_signal, weight_background, event_q
    gc.collect()

print("Ratio normalizations:", RATIO_NORMALIZATION)
print("Compressed q quantiles:", np.quantile(COMPRESSED_Q, [0, .01, .5, .99, 1]))

# The frozen event networks are no longer needed after compression. Free
# their Torch/ONNX GPU allocations before the large statistic flow trains.
reference_flow = None
ratio_models = None
PRESEL_model = None
PRESEL_scaler = None
model_proto = None
scaler = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released frozen Exercise 5 event networks.")


### Validate the compression over the full design interval

Exercise 5 checked one Asimov displacement. Here the unbinned and compressed expected statistics are compared for several generating and tested values across $[0,3]$. This validates the numerical acceleration before it is used half a million times.


In [ ]:
compression_validation = pd.DataFrame({
    "mu_true": compression_truth,
    "mu_test": compression_test,
    "t_unbinned": compression_unbinned,
    "t_compressed": compression_binned,
})
compression_validation["absolute_difference"] = (
    compression_validation["t_compressed"]
    - compression_validation["t_unbinned"]
)
compression_validation["relative_difference"] = np.divide(
    compression_validation["absolute_difference"],
    compression_validation["t_unbinned"],
    out=np.zeros(len(compression_validation)),
    where=compression_validation["t_unbinned"] > 1.0e-8,
)
display(compression_validation.style.format(precision=6).hide(axis="index"))
max_relative = float(compression_validation["relative_difference"].abs().max())
max_absolute = float(compression_validation["absolute_difference"].abs().max())
print(f"Maximum relative/absolute change: {max_relative:.3%} / {max_absolute:.4g}")
if max_relative > 5.0e-3 and max_absolute > 2.0e-2:
    raise RuntimeError(
        "The 512-bin compression is not accurate to 0.5% across the scan. "
        "Increase TOY_Q_BINS."
    )


## Vectorized pseudo-experiment fits

For a binned pseudo-experiment with counts $n_j$, the constrained MLE solves

$$
0=\lambda_S-\sum_j n_j\frac{q_j}{1+\widehat\mu q_j},
\qquad \widehat\mu\geq0.
$$

JAX performs 16 bounded Newton steps for an entire toy batch. The statistic is then

$$
t_\mu=2\left[(\mu-\widehat\mu)\lambda_S
-\sum_j n_j\left\{\log(1+\mu q_j)-\log(1+\widehat\mu q_j)\right\}\right].
$$

Only one combined Poisson draw is needed per bin, with mean $\mu\lambda_S P_{S,j}+\lambda_B P_{B,j}$. The $2000\times512$ count matrix is discarded after each fit.


In [ ]:
COMPRESSED_Q_JAX = jnp.asarray(COMPRESSED_Q)


@jax.jit
def fit_compressed_toy_batch(counts, test_mu):
    counts = jnp.asarray(counts, dtype=jnp.float64)
    test_mu = jnp.asarray(test_mu, dtype=jnp.float64).reshape(-1)
    q_values = COMPRESSED_Q_JAX
    initial_mu = jnp.clip(
        (jnp.sum(counts, axis=1) - LAM_BKG) / LAM_SIG,
        0.0,
        TOY_MU_MAX,
    )
    score_at_zero = LAM_SIG - jnp.sum(counts * q_values, axis=1)

    def newton_step(_, mu):
        response = q_values / (1.0 + mu[:, None] * q_values)
        score = LAM_SIG - jnp.sum(counts * response, axis=1)
        information = jnp.sum(counts * response**2, axis=1)
        step = jnp.clip(
            score / jnp.maximum(information, 1.0e-12), -2.0, 2.0
        )
        return jnp.clip(mu - step, 0.0, TOY_MU_MAX)

    mu_hat = jax.lax.fori_loop(
        0, TOY_NEWTON_STEPS, newton_step, initial_mu
    )
    mu_hat = jnp.where(score_at_zero >= 0.0, 0.0, mu_hat)
    statistic = 2.0 * (
        (test_mu - mu_hat) * LAM_SIG
        - jnp.sum(
            counts * (
                jnp.log1p(test_mu[:, None] * q_values)
                - jnp.log1p(mu_hat[:, None] * q_values)
            ),
            axis=1,
        )
    )
    fitted_response = q_values / (1.0 + mu_hat[:, None] * q_values)
    fitted_score = LAM_SIG - jnp.sum(
        counts * fitted_response, axis=1
    )
    fitted_score = jnp.where(mu_hat == 0.0, 0.0, fitted_score)
    return mu_hat, jnp.maximum(statistic, 0.0), fitted_score


def fit_toy_batch_numpy(counts, test_mu):
    counts = np.asarray(counts)
    test_mu = np.asarray(test_mu, dtype=np.float64).reshape(-1)
    result = fit_compressed_toy_batch(counts, test_mu)
    mu_hat, t_mu, fitted_score = [
        np.asarray(values, dtype=np.float64) for values in result
    ]
    failed = (
        (mu_hat > 1.0e-10)
        & (mu_hat < TOY_MU_MAX - 1.0e-10)
        & (np.abs(fitted_score) > 1.0e-6)
    )
    if np.any(failed):
        # The score is monotone increasing in mu. A vectorized bisection
        # fallback makes rare Newton failures harmless without returning
        # to one-Minuit-fit-per-toy execution.
        failed_counts = counts[failed].astype(np.float64)
        lower = np.zeros(np.sum(failed), dtype=np.float64)
        upper = np.full(np.sum(failed), TOY_MU_MAX, dtype=np.float64)
        for _ in range(64):
            middle = 0.5 * (lower + upper)
            response = COMPRESSED_Q / (
                1.0 + middle[:, None] * COMPRESSED_Q
            )
            middle_score = LAM_SIG - np.sum(
                failed_counts * response, axis=1
            )
            move_lower = middle_score < 0.0
            lower = np.where(move_lower, middle, lower)
            upper = np.where(move_lower, upper, middle)
        repaired_mu = 0.5 * (lower + upper)
        mu_hat[failed] = repaired_mu
        failed_test_mu = test_mu[failed]
        repaired_t = 2.0 * (
            (failed_test_mu - repaired_mu) * LAM_SIG
            - np.sum(
                failed_counts
                * (
                    np.log1p(failed_test_mu[:, None] * COMPRESSED_Q)
                    - np.log1p(repaired_mu[:, None] * COMPRESSED_Q)
                ),
                axis=1,
            )
        )
        t_mu[failed] = np.maximum(repaired_t, 0.0)
        repaired_response = COMPRESSED_Q / (
            1.0 + repaired_mu[:, None] * COMPRESSED_Q
        )
        fitted_score[failed] = LAM_SIG - np.sum(
            failed_counts * repaired_response, axis=1
        )
    if np.any(mu_hat >= TOY_MU_MAX - 1.0e-10):
        raise RuntimeError(
            "A toy MLE reached TOY_MU_MAX; increase the fit bound."
        )
    return mu_hat, t_mu, fitted_score


# Compile once and check that the fitted score is small away from the boundary.
_rng = np.random.default_rng(SEED + 200)
_mu = _rng.uniform(*MU_RANGE, size=8)
_mean = (
    _mu[:, None] * LAM_SIG * HNDE_SIGNAL_PROBABILITY[None, :]
    + LAM_BKG * HNDE_BACKGROUND_PROBABILITY[None, :]
)
_counts = _rng.poisson(_mean)
_muhat, _tmu, _score = fit_toy_batch_numpy(_counts, _mu)
print("Toy-kernel smoke test:")
print("  mu_true:", np.round(_mu, 3))
print("  mu_hat: ", np.round(_muhat, 3))
print("  t_mu:  ", np.round(_tmu, 3))
print(f"  max interior score residual: {np.max(np.abs(_score)):.3e}")
del _rng, _mu, _mean, _counts, _muhat, _tmu, _score


## 1. Generate 500,000 hNDE toys over $\mu\sim U(0,3)$

The ensemble is divided *before training*: 400,000 rows train/validate the conditional flow, and the remaining 100,000 rows train the first matched residual. This avoids asking a classifier to correct a flow on the same pseudo-experiments that fitted the flow.


In [ ]:
hnde_toys = run_cached_toy_ensemble(
    cache_dir=CACHE_DIR / f"hnde_uniform_{N_HNDE_TOYS}",
    n_toys=N_HNDE_TOYS,
    batch_size=TOY_BATCH_SIZE,
    seed=SEED + 300,
    mu_range=MU_RANGE,
    signal_probability=HNDE_SIGNAL_PROBABILITY,
    background_probability=HNDE_BACKGROUND_PROBABILITY,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    likelihood_q=COMPRESSED_Q,
    fit_batch=fit_toy_batch_numpy,
    fit_fingerprint=TOY_FIT_FINGERPRINT,
)
print(pd.DataFrame({
    "mu": hnde_toys["mu"],
    "mu_hat": hnde_toys["mu_hat"],
    "t_mu": hnde_toys["t_mu"],
    "n_events": hnde_toys["n_events"],
}).describe(percentiles=[.01, .5, .95, .99]).to_string())
print(
    "Maximum fitted-score residual:",
    f"{np.max(np.abs(hnde_toys['fitted_score'])):.3e}",
)

split_rng = np.random.default_rng(SEED + 301)
toy_order = split_rng.permutation(N_HNDE_TOYS)
flow_indices = toy_order[:N_FLOW_TOYS]
ratio1_indices = toy_order[N_FLOW_TOYS:]
flow_mu = hnde_toys["mu"][flow_indices].astype(np.float32)
flow_y = np.log(
    hnde_toys["t_mu"][flow_indices].astype(np.float64) + T_OFFSET
).astype(np.float32)
ratio1_mu = hnde_toys["mu"][ratio1_indices].astype(np.float32)
ratio1_y_positive = np.log(
    hnde_toys["t_mu"][ratio1_indices].astype(np.float64) + T_OFFSET
).astype(np.float32)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
axes[0].hexbin(
    hnde_toys["mu"], hnde_toys["mu_hat"],
    gridsize=80, bins="log", mincnt=1, cmap="viridis",
)
axes[0].plot(MU_RANGE, MU_RANGE, "w--", lw=1.4)
axes[0].set(xlabel=r"$\mu_{\rm true}$", ylabel=r"$\widehat\mu$",
            title="Amortized hNDE pseudo-experiments")
for low, high, color in [(0.0,.25,"C0"),(.75,1.0,"C1"),(1.75,2.0,"C2"),(2.75,3.0,"C3")]:
    mask = (hnde_toys["mu"] >= low) & (hnde_toys["mu"] < high)
    values = hnde_toys["t_mu"][mask]
    edges = np.linspace(0, np.quantile(values, .995), 60)
    axes[1].hist(values, bins=edges, density=True, histtype="step", lw=1.8,
                 color=color, label=rf"$\mu\in[{low:g},{high:g})$")
x = np.linspace(0.001, axes[1].get_xlim()[1], 400)
axes[1].plot(x, chi2.pdf(x, df=1), "k--", lw=1.3, label=r"$\chi^2_1$")
axes[1].set(xlabel=r"$t_{\mu_{\rm true}}$", ylabel="Density",
            title="The statistic is not assumed pivotal")
axes[1].legend(fontsize=8)
fig.tight_layout()
export_exercise11_figure(fig, "hnde_amortized_toys")
plt.show()


## 2. Train the conditional quadratic-spline reference

The monotone transformation

$$
y=\log(t_\mu+\epsilon),\qquad \epsilon=10^{-6},
$$

makes the positive, long-tailed statistic easier to model on the real line. All density ratios are trained in $(\mu,y)$ coordinates. The transformation is one-to-one for $t\geq0$, so its Jacobian cancels in the ratios and its conditional quantiles transform back exactly.


In [ ]:
statistic_flow = train_spline_flow(
    flow_y[:, None],
    context=flow_mu[:, None],
    checkpoint=MODEL_DIR / "q_phi_y_given_mu.pt",
    model_config=STATISTIC_FLOW_MODEL_CONFIG,
    training_config=STATISTIC_FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 400,
    load_if_available=LOAD_IF_AVAILABLE,
)
flow_config_mismatch = {
    name: (statistic_flow["config"].get(name), expected)
    for name, expected in STATISTIC_FLOW_MODEL_CONFIG.items()
    if statistic_flow["config"].get(name) != expected
}
if flow_config_mismatch:
    raise RuntimeError(
        "The loaded statistic-flow checkpoint has stale architecture: "
        f"{flow_config_mismatch}. Bump RUN_TAG."
    )
print("Conditional statistic flow:", statistic_flow["checkpoint"])


## 3. First matched correction: flow $\rightarrow$ hNDE toys

For every held-out hNDE pair $(\mu_i,y_i^+)$, draw $y_i^-\sim q_\phi(y\mid\mu_i)$. The two classifier rows share exactly the same $\mu_i$ and remain in the same train/validation group. The arithmetic mean of four classifier odds estimates $r_1=p_{\rm H}/q_\phi$.

The flow has real support while physical $y$ obeys $y\geq\log\epsilon$. Reference draws below that boundary are rejection-sampled. Their fraction is printed as a direct support diagnostic.


In [ ]:
Y_MIN = float(np.log(T_OFFSET))
ratio1_y_negative, ratio1_rejection = sample_truncated_spline_flow(
    statistic_flow,
    ratio1_mu[:, None],
    lower_bound=Y_MIN,
    seed=SEED + 500,
)
ratio1_positive = np.column_stack([ratio1_mu, ratio1_y_positive])
ratio1_negative = np.column_stack([ratio1_mu, ratio1_y_negative])
paired_ids_1 = np.arange(N_RATIO1_TOYS, dtype=np.int64)
ratio1_ensemble = []
for member in range(RATIO_ENSEMBLE_SIZE):
    print("\n" + "=" * 76)
    print(f"Training hNDE residual member {member + 1}/{RATIO_ENSEMBLE_SIZE}")
    print("=" * 76)
    ratio1_ensemble.append(
        train_ratio_classifier(
            ratio1_positive,
            ratio1_negative,
            checkpoint=RATIO1_MODEL_DIR / f"r1_member{member}.pt",
            model_config=CORRECTION_MODEL_CONFIG,
            training_config=CORRECTION_TRAINING_CONFIG,
            device=device,
            seed=SEED + 510 + 100 * member,
            load_if_available=LOAD_IF_AVAILABLE,
            paired_group_ids=paired_ids_1,
        )
    )
print(f"Unphysical flow-reference rejection fraction: {ratio1_rejection:.4%}")
assert all(
    pack["history"].get("split_strategy") == "paired_groups"
    for pack in ratio1_ensemble
)

fig, ax = plt.subplots(figsize=(7.0, 4.5))
for member, pack in enumerate(ratio1_ensemble):
    history = pack.get("history", {})
    ax.plot(history.get("validation", []), label=f"member {member}")
ax.set(xlabel="Epoch", ylabel="Validation BCE",
       title="First conditional-ratio ensemble")
ax.grid(alpha=.25)
ax.legend(ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "first_ratio_training")
plt.show()


## 4. Normalize the first hybrid density and compute its primitive

A finite classifier does not guarantee $\int q_\phi r_1\,dy=1$ at every $\mu$. Since the statistic is one-dimensional, we evaluate

$$
F_1(y\mid\mu)=
\frac{\int_{\log\epsilon}^{y}q_\phi(u\mid\mu)r_1(u,\mu)\,du}
{\int_{\log\epsilon}^{\infty}q_\phi(u\mid\mu)r_1(u,\mu)\,du}
$$

by deterministic cumulative trapezoidal quadrature. This is the numerical primitive of the corrected density; after multiplying by a neural ratio, it is no longer the flow's analytic CDF. The raw flow mass $C_\phi(\mu)$ on the physical domain is integrated separately, so $\widetilde Z_1=Z_1/C_\phi$ is the correct normalizer for rejection-sampled flow draws. The same conservative log-ratio bound is used in this quadrature and in the second-stage importance weights.


In [ ]:
MU_DENSITY_GRID = np.linspace(*MU_RANGE, QUADRATURE_MU_POINTS)
Y_MAX = float(np.quantile(np.concatenate([flow_y, ratio1_y_positive]), .99999) + 2.5)
Y_GRID = np.linspace(Y_MIN, Y_MAX, QUADRATURE_Y_POINTS)

flow_physical_grid = conditional_density_grid(
    statistic_flow,
    [],
    MU_DENSITY_GRID,
    Y_GRID,
)
hybrid1_grid = conditional_density_grid(
    statistic_flow,
    [ratio1_ensemble],
    MU_DENSITY_GRID,
    Y_GRID,
    max_abs_log_ratio=LOG_RATIO_CLIP,
)
hybrid1_quantile_y = conditional_quantiles(
    hybrid1_grid["cdf"], Y_GRID, QUANTILE_LEVELS
)
hybrid1_quantile_t = np.maximum(
    np.exp(hybrid1_quantile_y) - T_OFFSET, 0.0
)
hybrid1_log_normalization_truncated = (
    hybrid1_grid["log_normalization"]
    - flow_physical_grid["log_normalization"]
)
flow_physical_mass = np.exp(flow_physical_grid["log_normalization"])
if np.any((flow_physical_mass <= 0.0) | (flow_physical_mass > 1.01)):
    raise RuntimeError("Invalid numerical physical-support mass.")
quadrature_rejection = 1.0 - float(np.mean(flow_physical_mass))
print(
    "Physical flow mass C_phi(mu) range:",
    flow_physical_mass.min(), flow_physical_mass.max(),
)
print(
    "Flow rejection: observed / quadrature =",
    f"{ratio1_rejection:.4%} / {quadrature_rejection:.4%}",
)
if abs(ratio1_rejection - quadrature_rejection) > 5.0e-3:
    raise RuntimeError(
        "Rejection sampling and the numerical physical-support mass "
        "disagree; enlarge or refine Y_GRID."
    )
print(
    "log Z1_tilde(mu) quantiles:",
    np.quantile(hybrid1_log_normalization_truncated, [0, .01, .5, .99, 1]),
)
flow_edge_ratio = np.max(
    flow_physical_grid["density"][:, -1]
    / np.max(flow_physical_grid["density"], axis=1)
)
print("Raw-flow upper-grid density / peak:", flow_edge_ratio)
if flow_edge_ratio > 1.0e-5:
    raise RuntimeError(
        "The raw-flow quadrature truncates visible upper-tail density; "
        "increase Y_MAX."
    )
print(
    "r1 quadrature logit range / clipped fraction:",
    hybrid1_grid["ratio_log_range"][0],
    f"{hybrid1_grid['ratio_clip_fraction'][0]:.4%}",
)
hybrid1_edge_ratio = np.max(
    hybrid1_grid["density"][:, -1]
    / np.max(hybrid1_grid["density"], axis=1)
)
print("Upper-grid density / peak (worst mu):", hybrid1_edge_ratio)
if hybrid1_edge_ratio > 1.0e-5:
    raise RuntimeError(
        "The quadrature grid truncates visible upper-tail density; "
        "increase Y_MAX before trusting its CDFs."
    )


## 5. Build two independent simulator templates

Exercise 5 saved ratios on its final, held-out evaluation reservoirs—background first, then signal. These events were never used to train PRESEL, the reference flow, or either event-level density ratio. We randomly divide each process reservoir in half:

- the **calibration template** generates the 100,000 toys used for $r_2$;
- the **audit template** generates a separate 100,000-toy LF2I diagnostic.

Both templates histogram the learned Exercise 5 event score into the same $\log q$ bins. Crucially, simulator toys are still **fitted with `COMPRESSED_Q` from the hNDE model**. Replacing it by the simulator-template ratio would silently replace the approximate likelihood by truth and erase the calibration problem. Exercise 5 did not persist its five-million-event ratio normalizers, so the reconstructed reference draw induces a negligible constant score-scale fluctuation; its large size suppresses that Monte Carlo effect, and the simulator correction calibrates the resulting frozen statistic itself.


In [ ]:
simulator_templates = simulator_templates_from_exercise5(
    weights_path=HYBRID_DENSITY_DIR / "weights_asimov.npy",
    ratio_signal_path=HYBRID_DENSITY_DIR / "ratio_signal_asimov.npy",
    ratio_background_path=HYBRID_DENSITY_DIR / "ratio_background_asimov.npy",
    log_q_edges=LOG_Q_EDGES,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    seed=SEED + 600,
)
SIM_CAL_SIGNAL_PROBABILITY = simulator_templates[
    "calibration_signal_probability"
]
SIM_CAL_BACKGROUND_PROBABILITY = simulator_templates[
    "calibration_background_probability"
]
SIM_AUDIT_SIGNAL_PROBABILITY = simulator_templates["audit_signal_probability"]
SIM_AUDIT_BACKGROUND_PROBABILITY = simulator_templates[
    "audit_background_probability"
]
print(
    f"Held-out simulator events: background={simulator_templates['n_background']:,}, "
    f"signal={simulator_templates['n_signal']:,}"
)
print(
    "Held-out weight/yield closure: "
    f"signal={simulator_templates['signal_yield_closure']:+.3%}, "
    f"background={simulator_templates['background_yield_closure']:+.3%}"
)
print(
    "Calibration/audit template L1 differences: "
    f"signal={np.sum(np.abs(SIM_CAL_SIGNAL_PROBABILITY - SIM_AUDIT_SIGNAL_PROBABILITY)):.4f}, "
    f"background={np.sum(np.abs(SIM_CAL_BACKGROUND_PROBABILITY - SIM_AUDIT_BACKGROUND_PROBABILITY)):.4f}"
)


## 6. Generate 100,000 simulator-calibration pseudo-experiments

The generating bin probabilities now come from the independent simulator template. The fitted statistic remains the same frozen hNDE profile-likelihood ratio, exactly as it would for real data.


In [ ]:
simulator_calibration_toys = run_cached_toy_ensemble(
    cache_dir=CACHE_DIR / f"simulator_calibration_{N_SIMULATOR_CALIBRATION_TOYS}",
    n_toys=N_SIMULATOR_CALIBRATION_TOYS,
    batch_size=TOY_BATCH_SIZE,
    seed=SEED + 700,
    mu_range=MU_RANGE,
    signal_probability=SIM_CAL_SIGNAL_PROBABILITY,
    background_probability=SIM_CAL_BACKGROUND_PROBABILITY,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    likelihood_q=COMPRESSED_Q,
    fit_batch=fit_toy_batch_numpy,
    fit_fingerprint=TOY_FIT_FINGERPRINT,
)
print(pd.DataFrame({
    "mu": simulator_calibration_toys["mu"],
    "mu_hat": simulator_calibration_toys["mu_hat"],
    "t_mu": simulator_calibration_toys["t_mu"],
}).describe(percentiles=[.01, .5, .95, .99]).to_string())


## 7. Second matched correction: hNDE hybrid $\rightarrow$ simulator

For each simulator pair $(\mu_i,y_i^+)$, draw $y_i^-\sim\widetilde q_\phi(y\mid\mu_i)$ from the physical, rejection-sampled flow. The negative event receives

$$
w_i=\frac{r_1(y_i^-,\mu_i)}{\widetilde Z_1(\mu_i)},
\qquad \widetilde Z_1=Z_1/C_\phi.
$$

The weighted BCE normalizes positive and negative classes separately in every train/validation split. Thus both effective classes retain the same proposal $\pi(\mu)$, and the odds estimate the **conditional** correction $r_2=p_{\rm sim}/p_1$, rather than absorbing an unwanted $\mu$-marginal factor.


In [ ]:
ratio2_mu = simulator_calibration_toys["mu"].astype(np.float32)
ratio2_y_positive = np.log(
    simulator_calibration_toys["t_mu"].astype(np.float64) + T_OFFSET
).astype(np.float32)
ratio2_y_negative, ratio2_rejection = sample_truncated_spline_flow(
    statistic_flow,
    ratio2_mu[:, None],
    lower_bound=Y_MIN,
    seed=SEED + 800,
)
ratio2_positive = np.column_stack([ratio2_mu, ratio2_y_positive])
ratio2_negative = np.column_stack([ratio2_mu, ratio2_y_negative])

log_r1_negative = np.clip(
    ratio_classifier_ensemble_logit(ratio1_ensemble, ratio2_negative),
    -LOG_RATIO_CLIP,
    LOG_RATIO_CLIP,
)
log_z1_negative = np.interp(
    ratio2_mu,
    MU_DENSITY_GRID,
    hybrid1_log_normalization_truncated,
)
log_negative_weights_2 = log_r1_negative - log_z1_negative
log_negative_weights_2 -= (
    logsumexp(log_negative_weights_2)
    - np.log(len(log_negative_weights_2))
)
negative_weights_2 = np.exp(log_negative_weights_2)
negative_ess = importance_effective_sample_size(negative_weights_2)
print(f"Second-stage flow rejection fraction: {ratio2_rejection:.4%}")
print(
    f"First-hybrid negative-class ESS: {negative_ess:,.0f}/"
    f"{len(negative_weights_2):,} ({negative_ess/len(negative_weights_2):.1%})"
)
print(
    "Negative-weight quantiles:",
    np.quantile(negative_weights_2, [0, .001, .01, .5, .99, .999, 1]),
)

paired_ids_2 = np.arange(N_SIMULATOR_CALIBRATION_TOYS, dtype=np.int64)
ratio2_ensemble = []
for member in range(RATIO_ENSEMBLE_SIZE):
    print("\n" + "=" * 76)
    print(f"Training simulator residual member {member + 1}/{RATIO_ENSEMBLE_SIZE}")
    print("=" * 76)
    ratio2_ensemble.append(
        train_ratio_classifier(
            ratio2_positive,
            ratio2_negative,
            checkpoint=RATIO2_MODEL_DIR / f"r2_member{member}.pt",
            model_config=CORRECTION_MODEL_CONFIG,
            training_config=CORRECTION_TRAINING_CONFIG,
            device=device,
            seed=SEED + 810 + 100 * member,
            load_if_available=LOAD_IF_AVAILABLE,
            paired_group_ids=paired_ids_2,
            positive_weights=np.ones(N_SIMULATOR_CALIBRATION_TOYS),
            negative_weights=negative_weights_2,
        )
    )
assert all(pack["history"].get("weighted_bce") for pack in ratio2_ensemble)

fig, ax = plt.subplots(figsize=(7.0, 4.5))
for member, pack in enumerate(ratio2_ensemble):
    ax.plot(pack.get("history", {}).get("validation", []),
            label=f"member {member}")
ax.set(xlabel="Epoch", ylabel="Weighted validation BCE",
       title="Simulator-correction ensemble")
ax.grid(alpha=.25)
ax.legend(ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "second_weighted_ratio_training")
plt.show()


## 8. Compare conditional primitives and quantiles

We now integrate $q_\phi r_1$ and $q_\phi r_1r_2$ on the same dense one-dimensional grid. The first is the hNDE-level hybrid distribution; the second is simulator-calibrated.

A continuous proposal never draws exactly $\mu=0$ or $3$. We therefore add explicit endpoint ensembles. At $\mu=0$, this is essential: the constraint $\widehat\mu\geq0$ creates a point mass at $t_0=0$, which an ordinary continuous flow and absolutely continuous ratios cannot represent. Endpoint critical values use the conservative higher empirical order statistic from independent calibration anchors, while the learned density describes the interior.


In [ ]:
hybrid2_grid = conditional_density_grid(
    statistic_flow,
    [ratio1_ensemble, ratio2_ensemble],
    MU_DENSITY_GRID,
    Y_GRID,
    max_abs_log_ratio=LOG_RATIO_CLIP,
)
hybrid2_quantile_y = conditional_quantiles(
    hybrid2_grid["cdf"], Y_GRID, QUANTILE_LEVELS
)
hybrid2_quantile_t = np.maximum(
    np.exp(hybrid2_quantile_y) - T_OFFSET, 0.0
)
print(
    "log Z2(mu) quantiles:",
    np.quantile(hybrid2_grid["log_normalization"], [0, .01, .5, .99, 1]),
)
hybrid2_edge_ratio = np.max(
    hybrid2_grid["density"][:, -1]
    / np.max(hybrid2_grid["density"], axis=1)
)
print("Calibrated upper-grid density / peak:", hybrid2_edge_ratio)
if hybrid2_edge_ratio > 1.0e-5:
    raise RuntimeError(
        "The calibrated quadrature grid truncates visible upper-tail "
        "density; increase Y_MAX."
    )
for stage, (log_range, clipped) in enumerate(zip(
    hybrid2_grid["ratio_log_range"],
    hybrid2_grid["ratio_clip_fraction"],
), start=1):
    print(
        f"r{stage} quadrature logit range / clipped fraction:",
        log_range, f"{clipped:.4%}",
    )

anchor_hnde = {}
anchor_simulator = {}
for anchor_mu in ANCHOR_MUS:
    key = f"mu_{anchor_mu:g}".replace(".", "p")
    anchor_hnde[anchor_mu] = run_cached_toy_ensemble(
        cache_dir=CACHE_DIR / f"anchor_hnde_{key}_{N_ANCHOR_TOYS}",
        n_toys=N_ANCHOR_TOYS,
        batch_size=TOY_BATCH_SIZE,
        seed=SEED + 900 + int(100 * anchor_mu),
        mu_range=MU_RANGE,
        fixed_mu=float(anchor_mu),
        signal_probability=HNDE_SIGNAL_PROBABILITY,
        background_probability=HNDE_BACKGROUND_PROBABILITY,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        likelihood_q=COMPRESSED_Q,
        fit_batch=fit_toy_batch_numpy,
        fit_fingerprint=TOY_FIT_FINGERPRINT,
    )
    anchor_simulator[anchor_mu] = run_cached_toy_ensemble(
        cache_dir=CACHE_DIR / f"anchor_simulator_calibration_{key}_{N_ANCHOR_TOYS}",
        n_toys=N_ANCHOR_TOYS,
        batch_size=TOY_BATCH_SIZE,
        seed=SEED + 910 + int(100 * anchor_mu),
        mu_range=MU_RANGE,
        fixed_mu=float(anchor_mu),
        signal_probability=SIM_CAL_SIGNAL_PROBABILITY,
        background_probability=SIM_CAL_BACKGROUND_PROBABILITY,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        likelihood_q=COMPRESSED_Q,
        fit_batch=fit_toy_batch_numpy,
        fit_fingerprint=TOY_FIT_FINGERPRINT,
    )

hybrid1_quantile_t_anchored = hybrid1_quantile_t.copy()
hybrid2_quantile_t_anchored = hybrid2_quantile_t.copy()
for anchor_mu in ANCHOR_MUS:
    grid_index = int(np.argmin(np.abs(MU_DENSITY_GRID - anchor_mu)))
    hybrid1_quantile_t_anchored[grid_index] = np.quantile(
        anchor_hnde[anchor_mu]["t_mu"],
        QUANTILE_LEVELS,
        method="higher",
    )
    hybrid2_quantile_t_anchored[grid_index] = np.quantile(
        anchor_simulator[anchor_mu]["t_mu"],
        QUANTILE_LEVELS,
        method="higher",
    )

index_95 = int(np.flatnonzero(np.isclose(QUANTILE_LEVELS, .95))[0])
critical_hnde_grid = hybrid1_quantile_t_anchored[:, index_95]
critical_simulator_grid = hybrid2_quantile_t_anchored[:, index_95]
critical_hnde = PchipInterpolator(
    MU_DENSITY_GRID, critical_hnde_grid, extrapolate=False
)
critical_simulator = PchipInterpolator(
    MU_DENSITY_GRID, critical_simulator_grid, extrapolate=False
)
np.savez_compressed(
    CACHE_DIR / "conditional_quantiles.npz",
    mu=MU_DENSITY_GRID,
    levels=QUANTILE_LEVELS,
    hnde=hybrid1_quantile_t_anchored,
    simulator_calibrated=hybrid2_quantile_t_anchored,
)

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.8))
colors = plt.cm.viridis(np.linspace(.12, .9, len(QUANTILE_LEVELS)))
for level, color, before, after in zip(
    QUANTILE_LEVELS, colors,
    hybrid1_quantile_t_anchored.T,
    hybrid2_quantile_t_anchored.T,
):
    linewidth = 2.8 if np.isclose(level, .95) else 1.25
    axes[0].plot(MU_DENSITY_GRID, before, color=color, ls="--", lw=linewidth,
                 label=rf"{level:.0%}" if not np.isclose(level, .95) else rf"{level:.0%} hNDE")
    axes[0].plot(MU_DENSITY_GRID, after, color=color, lw=linewidth,
                 label=rf"{level:.0%} calibrated" if np.isclose(level, .95) else None)
    axes[1].plot(MU_DENSITY_GRID, after - before, color=color, lw=linewidth,
                 label=rf"{level:.0%}")
axes[0].set(xlabel=r"$\mu$", ylabel=r"conditional quantile of $t_\mu$",
            title="Before and after simulator correction")
axes[1].axhline(0.0, color="black", ls=":", lw=1)
axes[1].set(xlabel=r"$\mu$", ylabel="calibrated − hNDE quantile",
            title="Calibration displacement")
for ax in axes:
    ax.grid(alpha=.2)
    ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "conditional_quantile_corrections")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.0), sharex=True, sharey=True)
for ax, mu_value in zip(axes.flat, [0.25, 1.0, 2.0, 3.0]):
    index = int(np.argmin(np.abs(MU_DENSITY_GRID - mu_value)))
    t_grid = np.maximum(np.exp(Y_GRID) - T_OFFSET, 0.0)
    ax.plot(t_grid, hybrid1_grid["cdf"][index], ls="--", lw=2,
            label="hNDE hybrid")
    ax.plot(t_grid, hybrid2_grid["cdf"][index], lw=2,
            label="simulator-corrected")
    ax.axhline(.95, color="0.4", ls=":", lw=1)
    ax.set_xlim(0, max(8.0, float(critical_simulator_grid[index]) * 1.5))
    ax.set_title(rf"$\mu={mu_value:g}$")
    ax.grid(alpha=.2)
for ax in axes[-1]:
    ax.set_xlabel(r"$t_\mu$")
for ax in axes[:, 0]:
    ax.set_ylabel("Conditional CDF")
axes[0, 0].legend()
fig.tight_layout()
export_exercise11_figure(fig, "conditional_cdf_primitives")
plt.show()


## 9. LF2I-style independent coverage auditor

Freeze the final critical-value curve, then generate a completely unused simulator ensemble. Define

$$
W_i=\mathbb I\!\left[t_{\mu_i}(\mathcal D_i)\leq c_{0.95}(\mu_i)\right].
$$

A probabilistic network receives **only** $\mu_i$ and minimizes ordinary, unweighted BCE at the natural 95/5 class frequency. Its population target is

$$
a^*(\mu)=\Pr(W=1\mid\mu),
$$

the local coverage function. We also show direct equal-width-bin estimates with Wilson intervals, which prevent a too-smooth auditor from hiding localized failures. This is an empirical audit, not a mathematical proof: it resolves deviations only at the scale allowed by 100,000 simulator toys and the auditor's smoothness.


In [ ]:
simulator_audit_toys = run_cached_toy_ensemble(
    cache_dir=CACHE_DIR / f"simulator_audit_{N_SIMULATOR_AUDIT_TOYS}",
    n_toys=N_SIMULATOR_AUDIT_TOYS,
    batch_size=TOY_BATCH_SIZE,
    seed=SEED + 1000,
    mu_range=MU_RANGE,
    signal_probability=SIM_AUDIT_SIGNAL_PROBABILITY,
    background_probability=SIM_AUDIT_BACKGROUND_PROBABILITY,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    likelihood_q=COMPRESSED_Q,
    fit_batch=fit_toy_batch_numpy,
    fit_fingerprint=TOY_FIT_FINGERPRINT,
)
audit_mu = simulator_audit_toys["mu"].astype(np.float64)
audit_t = simulator_audit_toys["t_mu"].astype(np.float64)
audit_critical_hnde = critical_hnde(audit_mu)
audit_critical_simulator = critical_simulator(audit_mu)
covered_hnde = audit_t <= audit_critical_hnde
covered_simulator = audit_t <= audit_critical_simulator
print(f"Global hNDE-only coverage:       {covered_hnde.mean():.4%}")
print(f"Global simulator-calibrated coverage: {covered_simulator.mean():.4%}")

coverage_auditor = train_coverage_auditor(
    audit_mu,
    covered_simulator,
    checkpoint=MODEL_DIR / "coverage_auditor_95cl.pt",
    model_config=AUDITOR_MODEL_CONFIG,
    training_config=AUDITOR_TRAINING_CONFIG,
    device=device,
    seed=SEED + 1010,
    load_if_available=LOAD_IF_AVAILABLE,
)
auditor_grid = coverage_auditor_probability(
    coverage_auditor, MU_DENSITY_GRID
)
null_bce = -0.95 * np.log(.95) - 0.05 * np.log(.05)
print(f"Bernoulli(0.95) null BCE: {null_bce:.6f}")
print(
    "Auditor predicted-coverage range:",
    f"[{auditor_grid.min():.4%}, {auditor_grid.max():.4%}]",
)


In [ ]:
coverage_edges = np.linspace(*MU_RANGE, 21)
binned_before = binned_coverage(
    audit_mu, covered_hnde, edges=coverage_edges
)
binned_after = binned_coverage(
    audit_mu, covered_simulator, edges=coverage_edges
)

anchor_audit_rows = []
for anchor_mu in ANCHOR_MUS:
    key = f"mu_{anchor_mu:g}".replace(".", "p")
    anchor_audit = run_cached_toy_ensemble(
        cache_dir=CACHE_DIR / f"anchor_simulator_audit_{key}_{N_AUDIT_ANCHOR_TOYS}",
        n_toys=N_AUDIT_ANCHOR_TOYS,
        batch_size=TOY_BATCH_SIZE,
        seed=SEED + 1100 + int(100 * anchor_mu),
        mu_range=MU_RANGE,
        fixed_mu=float(anchor_mu),
        signal_probability=SIM_AUDIT_SIGNAL_PROBABILITY,
        background_probability=SIM_AUDIT_BACKGROUND_PROBABILITY,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        likelihood_q=COMPRESSED_Q,
        fit_batch=fit_toy_batch_numpy,
        fit_fingerprint=TOY_FIT_FINGERPRINT,
    )
    endpoint_critical = float(critical_simulator(anchor_mu))
    endpoint_covered = anchor_audit["t_mu"] <= endpoint_critical
    successes = int(endpoint_covered.sum())
    lower, upper = wilson_interval(successes, len(endpoint_covered))
    anchor_audit_rows.append({
        "mu": anchor_mu,
        "critical_value": endpoint_critical,
        "coverage": float(endpoint_covered.mean()),
        "wilson_lower": float(lower),
        "wilson_upper": float(upper),
        "zero_mass": float(np.mean(anchor_audit["t_mu"] <= 1.0e-12)),
    })
anchor_audit_table = pd.DataFrame(anchor_audit_rows)
display(anchor_audit_table.style.format(precision=5).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8.2, 5.2))
ax.axhline(.95, color="black", ls="--", lw=1.5, label="Nominal 95%")
ax.plot(MU_DENSITY_GRID, auditor_grid, color="C3", lw=2.3,
        label="BCE coverage auditor")
ax.errorbar(
    binned_before["center"], binned_before["coverage"],
    yerr=[
        binned_before["coverage"] - binned_before["lower"],
        binned_before["upper"] - binned_before["coverage"],
    ],
    fmt="o", ms=4, color="0.5", alpha=.75, label="hNDE-only cutoff",
)
ax.errorbar(
    binned_after["center"], binned_after["coverage"],
    yerr=[
        binned_after["coverage"] - binned_after["lower"],
        binned_after["upper"] - binned_after["coverage"],
    ],
    fmt="o", ms=5, color="C0", label="calibrated cutoff",
)
ax.errorbar(
    anchor_audit_table["mu"], anchor_audit_table["coverage"],
    yerr=[
        anchor_audit_table["coverage"] - anchor_audit_table["wilson_lower"],
        anchor_audit_table["wilson_upper"] - anchor_audit_table["coverage"],
    ],
    fmt="s", ms=6, color="C2", label="independent endpoint anchors",
)
ax.set(xlim=MU_RANGE, xlabel=r"true $\mu$", ylabel="Conditional coverage",
       title="Independent LF2I coverage audit")
ax.set_ylim(min(.90, binned_before["lower"].min() - .005), 1.005)
ax.grid(alpha=.2)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "lf2i_coverage_audit")
plt.show()

coverage_summary = pd.DataFrame({
    "method": ["hNDE hybrid", "simulator-corrected hybrid"],
    "global_coverage": [covered_hnde.mean(), covered_simulator.mean()],
    "minimum_binned_coverage": [
        binned_before["coverage"].min(), binned_after["coverage"].min()
    ],
    "maximum_binned_coverage": [
        binned_before["coverage"].max(), binned_after["coverage"].max()
    ],
})
display(coverage_summary.style.format(precision=5).hide(axis="index"))


## Interpretation

This exercise separates three roles that are often conflated:

- The frozen hNDE likelihood defines the statistic. Better event-level density ratios generally improve power and shorten the resulting confidence interval.
- Simulator calibration determines the conditional critical values. Even an imperfect statistic can give a valid test once its null distribution is correctly calibrated; approximation quality then primarily affects power rather than type-I error.
- The LF2I diagnostic is independent of construction. A flat auditor near 95%, supported by the binned Wilson intervals and endpoint anchors, means that no departure is statistically resolved at this audit's resolution. It does **not** prove exact coverage for a finite training budget.

The confidence set for observed data $\mathcal D_{\rm obs}$ is obtained by inversion:

$$
\mathcal C_{0.95}(\mathcal D_{\rm obs})
=\left\{\mu\in[0,3]:
t_\mu(\mathcal D_{\rm obs})\leq c_{0.95}(\mu)\right\}.
$$

The simulator template is itself a finite held-out Monte Carlo approximation. A production analysis would propagate that template uncertainty, enlarge the simulator calibration set, add nuisance parameters to the proposal, and reserve a final audit sample that is never reused after looking at its result.


## Suggested exercises

1. Compare the full-density construction with direct 95% conditional quantile regression (the canonical LF2I calibration branch).
2. Increase the simulator-calibration sample and study the convergence of $c_{0.95}(\mu)$ and the second-ratio weight ESS.
3. Replace the uniform proposal by a design with additional mass near boundaries, while preserving explicit importance factors.
4. Train a spike-and-slab model for the $t_0=0$ atom instead of treating $\mu=0$ with an empirical anchor.
5. Invert the learned critical-value curve for several observed pseudo-datasets and compare interval length before and after simulator calibration.
